In [3]:
# ============================================================
# 02_preprocessing.ipynb
# Préparation des données pour le ML
# ============================================================

# ── Cellule 1 : Configuration ────────────────────────────────
BASE_NAME = 'BIJOU'   # ← changer ici

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text
import os
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

SEUIL_MIN_JOURS   = 60    # minimum de jours pour entraîner
HORIZON_PREVISION = 30    # jours à prévoir
TEST_SPLIT_DAYS   = 30    # jours réservés pour le test

SERVER   = 'SALMAIKSOD\SAGE100'
DATABASE = 'Test'

conn_str = (
    f"mssql+pyodbc://sa:123456@{SERVER}/{DATABASE}"
    f"?driver=ODBC+Driver+17+for+SQL+Server"
    f"&TrustServerCertificate=yes"
)
engine = create_engine(conn_str, fast_executemany=True)
print(f"✅ Config OK — {BASE_NAME}")

✅ Config OK — BIJOU


In [4]:
# ── Cellule 2 : Rechargement données brutes ───────────────────
query = """
SELECT
    DateJour, AR_Ref, AR_Design,
    FA_CodeFamille, FA_Intitule,
    CL_No1, CL_Intitule1,
    DE_No, DE_Intitule,
    TotalEntree, TotalSortie,
    StockInitial, StockFinal, ValeurFinale
FROM Test.stock.VW_StockJoursAvecMvt
WHERE BaseName = :base
ORDER BY AR_Ref, DE_No, DateJour
"""

with engine.connect() as conn:
    df_raw = pd.read_sql(
        text(query), conn,
        params={'base': BASE_NAME},
        parse_dates=['DateJour']
    )

print(f"✅ {len(df_raw):,} lignes chargées")

✅ 88 lignes chargées


In [5]:
# ── Cellule 3 (remplacée) : Vérification que les séries sont complètes
# SP_GetStockJournalier gère déjà le report des jours sans mouvement
# On vérifie juste que c'est bien le cas

def verifier_completude(group):
    dates_attendues = pd.date_range(
        group['DateJour'].min(),
        group['DateJour'].max(),
        freq='D'
    )
    manquants = len(dates_attendues) - len(group)
    return manquants

manquants_par_combo = (
    df_raw.groupby(['AR_Ref', 'DE_No'])
    .apply(verifier_completude)
    .reset_index()
    .rename(columns={0: 'jours_manquants'})
)

total_manquants = manquants_par_combo['jours_manquants'].sum()

if total_manquants == 0:
    print("✅ Toutes les séries sont complètes (aucun jour manquant)")
    df_complete = df_raw.copy()
else:
    print(f"⚠️  {total_manquants} jours manquants détectés — complétion manuelle nécessaire")
    # Dans ce cas tu rappelles la procédure ou tu complètes manuellement
    df_complete = df_raw.copy()

print(f"   Dataset : {len(df_complete):,} lignes")

⚠️  1186 jours manquants détectés — complétion manuelle nécessaire
   Dataset : 88 lignes


In [6]:
# ── Cellule 4 : Filtrer les articles prévisibles ──────────────
nb_jours_par_combo = (
    df_raw.groupby(['AR_Ref', 'DE_No'])['DateJour']
    .count()
    .reset_index()
    .rename(columns={'DateJour': 'nb_jours_mvt'})
)

combos_valides = nb_jours_par_combo[
    nb_jours_par_combo['nb_jours_mvt'] >= SEUIL_MIN_JOURS
]

print(f"Combinaisons article+dépôt total    : {len(nb_jours_par_combo)}")
print(f"Combinaisons prévisibles (>= {SEUIL_MIN_JOURS}j) : {len(combos_valides)}")
print(f"Combinaisons exclues                : {len(nb_jours_par_combo) - len(combos_valides)}")

df_filtered = df_complete.merge(
    combos_valides[['AR_Ref', 'DE_No']],
    on=['AR_Ref', 'DE_No'],
    how='inner'
)

print(f"\n✅ Dataset filtré : {len(df_filtered):,} lignes sur {df_filtered['AR_Ref'].nunique()} articles")

Combinaisons article+dépôt total    : 46
Combinaisons prévisibles (>= 60j) : 0
Combinaisons exclues                : 46

✅ Dataset filtré : 0 lignes sur 0 articles


In [9]:
print(df_clean.columns.tolist())
print(df_clean.head())

['DateJour', 'AR_Design', 'FA_CodeFamille', 'FA_Intitule', 'CL_No1', 'CL_Intitule1', 'DE_Intitule', 'TotalEntree', 'TotalSortie', 'StockInitial', 'StockFinal', 'ValeurFinale']
Empty DataFrame
Columns: [DateJour, AR_Design, FA_CodeFamille, FA_Intitule, CL_No1, CL_Intitule1, DE_Intitule, TotalEntree, TotalSortie, StockInitial, StockFinal, ValeurFinale]
Index: []


In [7]:
# ── Cellule 5 : Détection et traitement des outliers ─────────
def traiter_outliers(group, col='TotalSortie', seuil_iqr=3.0):
    """
    Remplace les sorties aberrantes par la médiane glissante.
    Un outlier = valeur > Q3 + seuil_iqr * IQR
    """
    q1  = group[col].quantile(0.25)
    q3  = group[col].quantile(0.75)
    iqr = q3 - q1
    borne_haute = q3 + seuil_iqr * iqr
    
    n_outliers = (group[col] > borne_haute).sum()
    if n_outliers > 0:
        mediane = group[col].median()
        group.loc[group[col] > borne_haute, col] = mediane
    
    return group

print("⏳ Traitement des outliers...")

df_clean = (
    df_filtered.groupby(['AR_Ref', 'DE_No'], group_keys=False)
    .apply(lambda g: traiter_outliers(g, 'TotalSortie'))
    .reset_index(drop=True)
)

# Visualiser l'effet du traitement sur un article pilote
article_pilote = df_clean['AR_Ref'].value_counts().index[0]
df_pilote_avant = df_filtered[df_filtered['AR_Ref'] == article_pilote]
df_pilote_apres = df_clean[df_clean['AR_Ref'] == article_pilote]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(df_pilote_avant.groupby('DateJour')['TotalSortie'].sum(), color='#e53935', linewidth=1)
axes[0].set_title(f'Avant traitement outliers\nArticle : {article_pilote}', fontweight='bold')
axes[1].plot(df_pilote_apres.groupby('DateJour')['TotalSortie'].sum(), color='#01a82e', linewidth=1)
axes[1].set_title(f'Après traitement outliers\nArticle : {article_pilote}', fontweight='bold')
for ax in axes:
    ax.set_xlabel("Date"); ax.set_ylabel("Sorties")
plt.tight_layout()
plt.savefig(f'../outputs/{BASE_NAME}_06_outliers.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ Nettoyage terminé")

⏳ Traitement des outliers...


KeyError: 'AR_Ref'

In [ ]:
# ── Cellule 6 : Feature engineering ─────────────────────────
def ajouter_features(group):
    g = group.sort_values('DateJour').copy()
    
    # Features temporelles
    g['jour_semaine']  = g['DateJour'].dt.dayofweek     # 0=Lun, 6=Dim
    g['mois']          = g['DateJour'].dt.month
    g['semaine_annee'] = g['DateJour'].dt.isocalendar().week.astype(int)
    g['trimestre']     = g['DateJour'].dt.quarter
    g['est_weekend']   = (g['jour_semaine'] >= 5).astype(int)
    g['est_fin_mois']  = (g['DateJour'].dt.day >= 25).astype(int)
    
    # Lag features : sorties des J-1, J-7, J-14, J-30
    for lag in [1, 7, 14, 30]:
        g[f'sortie_lag_{lag}'] = g['TotalSortie'].shift(lag).fillna(0)
    
    # Moyennes mobiles
    g['ma7']  = g['TotalSortie'].shift(1).rolling(7,  min_periods=1).mean().fillna(0)
    g['ma14'] = g['TotalSortie'].shift(1).rolling(14, min_periods=1).mean().fillna(0)
    g['ma30'] = g['TotalSortie'].shift(1).rolling(30, min_periods=1).mean().fillna(0)
    
    # Tendance : moyenne 7j vs moyenne 30j
    g['tendance'] = (g['ma7'] - g['ma30']).fillna(0)
    
    # Stock features
    g['stock_lag1']      = g['StockFinal'].shift(1).fillna(method='bfill')
    g['ratio_stock_ma7'] = (g['stock_lag1'] / (g['ma7'] + 1)).fillna(1)
    
    return g

print("⏳ Feature engineering en cours...")

df_features = (
    df_clean.groupby(['AR_Ref', 'DE_No'], group_keys=False)
    .apply(ajouter_features)
    .reset_index(drop=True)
)

print(f"✅ Features créées")
print(f"   Colonnes disponibles ({len(df_features.columns)}) :")
print([c for c in df_features.columns])

In [ ]:
# ── Cellule 7 : Visualiser les features sur un article pilote ─
g_pilote = df_features[
    df_features['AR_Ref'] == article_pilote
].sort_values('DateJour')

fig, axes = plt.subplots(3, 1, figsize=(14, 12))

axes[0].plot(g_pilote['DateJour'], g_pilote['TotalSortie'],
             color='#e53935', linewidth=1, label='Sorties réelles')
axes[0].plot(g_pilote['DateJour'], g_pilote['ma7'],
             color='#ff9800', linewidth=2, linestyle='--', label='MA7')
axes[0].plot(g_pilote['DateJour'], g_pilote['ma30'],
             color='#b71c1c', linewidth=2, linestyle=':', label='MA30')
axes[0].set_title(f'Sorties + Moyennes mobiles — {article_pilote}', fontweight='bold')
axes[0].legend(); axes[0].set_ylabel("Unités")

axes[1].plot(g_pilote['DateJour'], g_pilote['StockFinal'],
             color='#12a6e0', linewidth=1.5)
axes[1].axhline(0, color='red', linestyle='--', linewidth=1, alpha=0.5)
axes[1].set_title('Évolution du stock final', fontweight='bold')
axes[1].set_ylabel("Stock")

axes[2].bar(g_pilote['DateJour'], g_pilote['tendance'],
            color=np.where(g_pilote['tendance'] >= 0, '#01a82e', '#e53935'),
            width=0.8)
axes[2].axhline(0, color='black', linewidth=0.5)
axes[2].set_title('Indicateur de tendance (MA7 - MA30)', fontweight='bold')
axes[2].set_ylabel("Tendance")

for ax in axes:
    ax.set_xlabel("Date")
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(f'../outputs/{BASE_NAME}_07_features_pilote.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cellule 8 : Split train / test ───────────────────────────
date_max    = df_features['DateJour'].max()
date_split  = date_max - pd.Timedelta(days=TEST_SPLIT_DAYS)

df_train = df_features[df_features['DateJour'] <= date_split]
df_test  = df_features[df_features['DateJour'] >  date_split]

print(f"Dataset total : {len(df_features):,} lignes")
print(f"Train : {len(df_train):,} lignes  ({df_train['DateJour'].min().date()} → {df_train['DateJour'].max().date()})")
print(f"Test  : {len(df_test):,}  lignes  ({df_test['DateJour'].min().date()}  → {df_test['DateJour'].max().date()})")

# Visualiser le split
sorties_train = df_train.groupby('DateJour')['TotalSortie'].sum()
sorties_test  = df_test.groupby('DateJour')['TotalSortie'].sum()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(sorties_train.index, sorties_train.values, color='#12a6e0', linewidth=1.2, label='Train')
ax.plot(sorties_test.index,  sorties_test.values,  color='#e53935',  linewidth=1.2, label='Test')
ax.axvline(date_split, color='black', linestyle='--', linewidth=1.5, label=f'Split ({date_split.date()})')
ax.set_title(f'Découpage Train / Test — {BASE_NAME}', fontweight='bold', fontsize=13)
ax.set_xlabel("Date"); ax.set_ylabel("Total sorties journalières")
ax.legend()
plt.tight_layout()
plt.savefig(f'../outputs/{BASE_NAME}_08_train_test_split.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cellule 9 : Sauvegarde du dataset préparé ─────────────────
os.makedirs('../outputs', exist_ok=True)

df_features.to_parquet(f'../outputs/{BASE_NAME}_dataset_features.parquet', index=False)

print(f"✅ Dataset sauvegardé : ../outputs/{BASE_NAME}_dataset_features.parquet")
print(f"   Taille             : {df_features.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"   Articles           : {df_features['AR_Ref'].nunique()}")
print(f"   Combinaisons       : {df_features.groupby(['AR_Ref','DE_No']).ngroups}")
print(f"   Colonnes           : {len(df_features.columns)}")
print(f"\n✅ Preprocessing terminé — passer au notebook 03")